# M05A: Token Counting & Cost Tracking

Every API call costs money. Before optimizing, you need to measure.

**Topics:**
- How tokenization works
- TokenCounter utility
- CostTracker utility
- Input vs output cost differences

---

## 🔧 Step 1: Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import tiktoken

load_dotenv(dotenv_path=Path("..") / ".env")

MODEL = "gpt-5-mini"

# Standard pricing per 1M tokens (illustrative)
# Verify current rates before production use
PRICING = {
    "gpt-5": {"input": 1.25, "output": 10.00},
    "gpt-5-mini": {"input": 0.25, "output": 2.00},
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60}
}


print(f"✅ Setup complete: Using {MODEL}!")

---

## 🎯 Why Tokens Matter

Every API call is billed by tokens — input tokens (what you send) and output tokens (what the model returns).  

To control costs, you need to measure them.

**This notebook teaches you how.**

---

## 📊 Part 1: Token Counting

Tokens are how LLMs measure text length and calculate costs.

In M01C we used `response.usage` to get token counts *after* an API call — what you actually paid.

`tiktoken` counts tokens *before* calling — same tokenizer OpenAI uses. Accurate for input, but output tokens remain unknown until the response arrives.

In [ ]:
class TokenCounter:
    """Count tokens for text."""
    
    def __init__(self, model=MODEL):
        try:
            self.tokenizer = tiktoken.encoding_for_model(model)
        except KeyError:
            print(f"⚠️  Model '{model}' not found. Using default encoding.")
            try:
                self.tokenizer = tiktoken.get_encoding("o200k_base")
            except Exception:
                self.tokenizer = tiktoken.get_encoding("cl100k_base")
    
    def count(self, text):
        """Count tokens in text."""
        return len(self.tokenizer.encode(text))


# --------------------------------------------------------------
print("✅ TokenCounter ready")

### Test Token Counting

In [ ]:
counter = TokenCounter()

texts = [
    "Hello!",
    "What is AI?",
    "Explain machine learning simply.",
    "Tokenization is straightforward."
]

print("🔢 TOKEN COUNTING DEMO")
print("="*60)
for text in texts:
    tokens = counter.tokenizer.encode(text)
    decoded = [counter.tokenizer.decode([t]) for t in tokens]
    print(f"{text:40} → {len(tokens)} tokens")
    print(f"{'':40}   {decoded}")
print("="*60)

---

## 💰 Part 2: Cost Tracking

Now that we can count tokens, let's convert them to dollars.

In [ ]:
class CostTracker:
    """Track cumulative API costs across turns."""
    
    def __init__(self, model=MODEL):
        self.model = model
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.token_counter = TokenCounter(model)
    
    def add_turn(self, user_message, assistant_message):
        """Record a turn and return its cost."""
        input_tokens = self.token_counter.count(user_message)
        output_tokens = self.token_counter.count(assistant_message)
        
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        
        return self._calculate_cost(input_tokens, output_tokens)
    
    def _calculate_cost(self, input_tokens, output_tokens):
        """Calculate cost for given token counts."""
        pricing = PRICING.get(self.model, PRICING["gpt-5-mini"])
        input_cost = (input_tokens / 1_000_000) * pricing["input"]
        output_cost = (output_tokens / 1_000_000) * pricing["output"]
        return input_cost + output_cost
    
    def get_total(self):
        """Get total cost so far."""
        return self._calculate_cost(self.total_input_tokens, self.total_output_tokens)


# --------------------------------------------------------------
print("✅ CostTracker ready")

### Test Cost Tracking

In [ ]:
tracker = CostTracker()

conversations = [
    ("What is Python?", "Python is a programming language."),
    ("What is it used for?", "Web dev, data science, AI, automation.")
]

print("💰 COST TRACKING DEMO")
print("="*60)
for user_msg, assistant_msg in conversations:
    cost = tracker.add_turn(user_msg, assistant_msg)
    input_tokens = tracker.token_counter.count(user_msg)
    output_tokens = tracker.token_counter.count(assistant_msg)
    print(f"Input: {input_tokens} tokens | Output: {output_tokens} tokens | Cost: ${cost:.6f}")
print(f"\nTotal cost: ${tracker.get_total():.6f}")
print("="*60)

### 💡 Key Insight

**Rule of thumb:** ~4 characters = 1 token. Use tiktoken for accurate counts — but for actual billing, check `response.usage` or your OpenAI dashboard.

**Output tokens cost 4–8× more than input tokens** — keep responses concise to save money. 

---

## 🎯 Key Takeaways

**🔢 Token Counting:**
- Use tiktoken for accurate counts, add fallback for new models
- Essential for cost calculation and context management

**💰 Cost Tracking:**
- Track input and output separately — output costs 4–8× more
- tiktoken estimates locally, `response.usage` gives actual billing

**The Flow:** Count tokens with tiktoken → Calculate cost using input/output rates → Track cumulative spend across turns

---

### 📍 Next Step

**M05B: Context Window Strategies** — Manage growing conversations with sliding windows, summarization, and a production-ready ConversationManager.

---

## 🔧 Troubleshooting

**Token counts don't match expectations?**
- Use `tiktoken.encoding_for_model()` for correct model encoding
- Special characters and emojis often use multiple tokens
- Never estimate with character counts—always use tiktoken

**Counts vary between local and API?**
- Ensure you're using the same model encoding locally and in API
- Check tiktoken version matches OpenAI's current tokenizer
- Formatting (whitespace) counts as tokens

**Costs higher than expected?**
- Output tokens cost more than input (4–8× depending on model)—keep responses concise
- Verify PRICING constants match current OpenAI pricing
- Long AI responses quickly increase costs

**Still having issues?**
- Copy any error message and paste it into ChatGPT, Claude, Gemini, or Grok — they're great at debugging
- Re-watch the lecture for this module
- Post to the Q&A with your error message and output

---